# Protein Distribution

This notebook explores the distribution of protein in the DisProt dataset, taking into 
account for their release date, existing PDB structures and whether they have been used 
for CAID evaluation.

In [1]:
# Setup the environment
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `~/.julia/dev/IDRBindingDataset.jl/scripts`


In [2]:
# Load the necessary packages.
# Important functions for the analysis are defined in the IDRBindingDataset package, rather
# than in this notebook, so that they can be easily reused in other contexts.
using CodecZlib
using DataFrames
using IDRBindingDataset
using JSON
using MIToS

## Load the DisProt dataset

We will first load the DisProt dataset (in JSON format), and for each protein—identified by
its UniProt accession number—we will retrieve the release version tag (`YYYY_MM`) of the
oldest annotated intrinsic disorder region (IDR). For that, we will first create a
DataFrame of IDRs, and then we will group the IDRs by their corresponding UniProt
accession number, and for each group we will retrieve the oldest region release tag.
This will lead to two `const` DataFrames we can later use throughout the notebook:
`IDRS` and `PROTEINS`.

### DisProt JSON format (API search output)

Source: DisProt API docs (`https://disprot.org/api`) and API search output (`https://disprot.org/api/search?...`).

According to the DisProt API docs for `api/search`, the response contains two top-level keys:
- `size`: number of protein entries returned
- `data`: array of protein entries

Each protein entry includes identifiers (`disprot_id`, `acc`), sequence-level metadata,
a protein-level `released` tag (`YYYY_MM`), and a `regions` array.

Each item in `regions` is one annotation record and includes fields such as:
- `region_id`, `start`, `end`
- region-level `released` (`YYYY_MM`)
- ontology labels (`term_id`, `term_name`, `term_namespace`)
- evidence/publication metadata (`ec_id`, `reference_id`, etc.)

Important: two region records can share the same `start` and `end` when they annotate
different concepts (for example structural state vs structural transition) on the same segment.

In [3]:
# Define the source DisProt file.
const DISPROT_JSON_PATH = joinpath(
    @__DIR__,
    "..",
    "data",
    "raw",
    "disprot",
    "disprot_regions.json.gz",
)

DISPROT_JSON_PATH

"/home/diego.zea/.julia/dev/IDRBindingDataset.jl/scripts/../data/raw/disprot/disprot_regions.json.gz"

In [4]:
# Load the compressed DisProt JSON payload and keep the protein-level entries.
const DISPROT_PAYLOAD = open(DISPROT_JSON_PATH) do io
    JSON.parse(read(GzipDecompressorStream(io), String))
end

const DISPROT_ENTRIES = DISPROT_PAYLOAD["data"]

println("Loaded $(length(DISPROT_ENTRIES)) proteins from $(DISPROT_JSON_PATH)")

Loaded 3199 proteins from /home/diego.zea/.julia/dev/IDRBindingDataset.jl/scripts/../data/raw/disprot/disprot_regions.json.gz


In [5]:
# Flatten the nested regions into an IDR-level DataFrame.
const IDRS = let rows = NamedTuple[]
    for protein in DISPROT_ENTRIES
        acc = protein["acc"]
        disprot_id = protein["disprot_id"]
        protein_released = protein["released"]

        for region in protein["regions"]
            push!(rows, (
                acc = acc,
                disprot_id = disprot_id,
                protein_released = protein_released,
                region_id = region["region_id"],
                term_id = region["term_id"],
                term_name = region["term_name"],
                term_namespace = region["term_namespace"],
                start = Int(region["start"]),
                stop = Int(region["end"]),
                released = region["released"],
            ))
        end
    end
    DataFrame(rows)
end

first(IDRS, 10)

Row,acc,disprot_id,protein_released,region_id,term_id,term_name,term_namespace,start,stop,released
,String,String,String,String,String,String,String,Int64,Int64,String
1,P03265,DP00003,2016_10,DP00003r002,IDPO:0000002,disorder,Structural state,294,334,2022_03
2,P03265,DP00003,2016_10,DP00003r004,IDPO:0000002,disorder,Structural state,454,464,2022_03
3,P49913,DP00004,2016_10,DP00004r001,IDPO:0000002,disorder,Structural state,134,170,2022_03
4,P49913,DP00004,2016_10,DP00004r002,IDPO:0000011,disorder to order,Structural transition,134,170,2022_03
5,P49913,DP00004,2016_10,DP00004r004,GO:0019835,cytolysis,Biological process,134,170,2022_06
6,P49913,DP00004,2016_10,DP00004r005,GO:1990000,amyloid fibril formation,Biological process,150,162,2022_12
7,P49913,DP00004,2016_10,DP00004r006,GO:1990000,amyloid fibril formation,Biological process,150,162,2022_12
8,P03045,DP00005,2016_10,DP00005r001,IDPO:0000002,disorder,Structural state,1,107,2022_03
9,P03045,DP00005,2016_10,DP00005r004,IDPO:0000002,disorder,Structural state,1,107,2022_03


In [6]:
# Group IDRs by UniProt accession and retain release information per protein.
const PROTEINS = let grouped = groupby(IDRS, :acc)
    combine(
        grouped,
        :disprot_id => first => :disprot_id,
        :protein_released => first => :protein_released,
        :released => (x -> minimum(parse_disprot_release.(x))) => :oldest_idr_release,
        :released => (x -> maximum(parse_disprot_release.(x))) => :most_recent_idr_release,
        nrow => :n_idrs,
    )
end

sort!(PROTEINS, :oldest_idr_release)
first(PROTEINS, 10)

Row,acc,disprot_id,protein_released,oldest_idr_release,most_recent_idr_release,n_idrs
,String,String,String,Date,Date,Int64
1,P03265,DP00003,2016_10,2022-03-01,2022-03-01,2
2,P49913,DP00004,2016_10,2022-03-01,2022-12-01,5
3,P03045,DP00005,2016_10,2022-03-01,2022-03-01,16
4,P00004,DP00006,2016_10,2022-03-01,2022-12-01,4
5,P27695,DP00007,2016_10,2022-03-01,2022-03-01,4
6,Q64693,DP00008,2016_10,2022-03-01,2022-03-01,5
7,P13569,DP00012,2016_10,2022-03-01,2024-06-01,18
8,P0DN86,DP00013,2016_10,2022-03-01,2022-03-01,4
9,P61926,DP00015,2016_10,2022-03-01,2022-03-01,19


In [7]:
# Print a compact summary so we can quickly sanity-check the generated tables.
println("Total IDRs: $(nrow(IDRS))")
println("Total proteins: $(nrow(PROTEINS))")
println("Oldest protein entry release among proteins: $(minimum(PROTEINS.protein_released))")
println("Most recent protein entry release among proteins: $(maximum(PROTEINS.protein_released))")
println("Oldest IDR release among proteins: $(minimum(PROTEINS.oldest_idr_release))")
println("Most recent IDR release among proteins: $(maximum(PROTEINS.most_recent_idr_release))")

Total IDRs: 13396
Total proteins: 3199
Oldest protein entry release among proteins: 2016_10
Most recent protein entry release among proteins: 2025_06
Oldest IDR release among proteins: 2022-03-01
Most recent IDR release among proteins: 2025-12-01


## UniProt to PDB Mapping from DisProt Cross References

This section builds a UniProt-to-PDB mapping using region-level DisProt cross
references. The final table has one row per unique `(acc, pdb)` pair.

In [ ]:
# Helper for extracting PDB cross references from one region.
function _collect_pdb_rows!(rows::DataFrame, acc::AbstractString, cross_refs)
    for ref in cross_refs
        db = uppercase(strip(String(get(ref, "db", ""))))
        db == "PDB" || continue
        pdb = uppercase(strip(String(get(ref, "id", ""))))
        occursin(r"^[0-9A-Z]{4}$", pdb) || continue
        push!(rows, (acc = String(acc), pdb = pdb))
    end
    return rows
end

const EMPTY_PDB_REF_ROWS = DataFrame(acc = String[], pdb = String[]);

Row,acc,pdb
,String,String


In [9]:
# Extract region-level PDB cross references.
const REGION_LEVEL_PDB_REFS = let rows = copy(EMPTY_PDB_REF_ROWS)
    for protein in DISPROT_ENTRIES
        acc = protein["acc"]
        for region in protein["regions"]
            _collect_pdb_rows!(rows, acc, get(region, "cross_refs", Any[]))
        end
    end
    sort!(rows, [:acc, :pdb])
end

println("Region-level UniProt-PDB rows (with repeated pairs across regions): $(nrow(REGION_LEVEL_PDB_REFS))")
first(REGION_LEVEL_PDB_REFS, 10)

Region-level UniProt-PDB rows (with repeated pairs across regions): 6730


Row,acc,pdb
,String,String
1,A0A024B7W1,6LD1
2,A0A024B7W1,6LD1
3,A0A024B7W1,7BSD
4,A0A090N8E9,4MI0
5,A0A097F4I8,7ELA
6,A0A097F4I8,7ELA
7,A0A0B4JDC9,6Q9M
8,A0A0B4JDC9,6Q9M
9,A0A0C4DGY3,6XT9


In [10]:
# Collapse repeated region references into one row per UniProt-PDB pair.
const UNIPROT_PDB_MAP = let rows = unique(REGION_LEVEL_PDB_REFS, [:acc, :pdb])
    sort!(rows, [:acc, :pdb])
end

println("Unique UniProt-PDB pairs: $(nrow(UNIPROT_PDB_MAP))")
first(UNIPROT_PDB_MAP, 10)

Unique UniProt-PDB pairs: 3782


Row,acc,pdb
,String,String
1,A0A024B7W1,6LD1
2,A0A024B7W1,7BSD
3,A0A090N8E9,4MI0
4,A0A097F4I8,7ELA
5,A0A0B4JDC9,6Q9M
6,A0A0C4DGY3,6XT9
7,A0A0G2KTI4,6NPJ
8,A0A0G2KTI4,6NPL
9,A0A0H2UWN8,7N10


In [14]:
# Show UniProt-PDB pairs statistics.
println("Total region-level PDB references: $(nrow(REGION_LEVEL_PDB_REFS))")
println("Unique UniProt-PDB pairs: $(nrow(UNIPROT_PDB_MAP))")
n_repeated_rows = nrow(REGION_LEVEL_PDB_REFS) - nrow(UNIPROT_PDB_MAP)
println("Repeated region rows collapsed: $(n_repeated_rows)")
println("Proteins with at least one PDB mapping: $(length(unique(UNIPROT_PDB_MAP.acc)))")

Total region-level PDB references: 6730
Unique UniProt-PDB pairs: 3782
Repeated region rows collapsed: 2948
Proteins with at least one PDB mapping: 2215
